# 2. Model and prompting-method selection

Calculate condition means from the blinded human ratings of the 50-item panel.
The supplied annotation workbook has two raters and 96 scores for 48 items.
Two generated items have no recorded scores.

Rank the observed means and show the recorded author choice. Cases differed
across conditions, so the means are descriptive rather than a matched-case
comparison.
The ratings retain the original pilot scope. Notebook 3 applies the selected
condition to the general-case taxonomy.


In [1]:
import json

from pandas import DataFrame, set_option

from warbias_kit.dataset_workflow import create_workspace, find_project, load_inputs
from warbias_kit.io import write_json
from warbias_kit.notebook_display import display

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = create_workspace("02_model_selection")
set_option("display.max_colwidth", None)
set_option("display.max_rows", 40)

## Rating records and coverage

Each row is one overall triplet-quality score from one anonymized rater. Join
scores by the recorded condition and blind IDs. Two generated items are unrated;
missing ratings remain missing.


In [2]:
ratings = DataFrame(inputs["pilot_human_ratings"])
ratings["score"] = ratings["score"].astype(float)
display(ratings[["blind_id", "candidate_id", "strategy", "rater_id", "score"]].head(12))
display(
    ratings.groupby("rater_id")
    .agg(items=("blind_id", "nunique"), ratings=("score", "size"))
    .reset_index()
)
display(
    ratings.groupby(["candidate_id", "strategy"])
    .agg(score_sum=("score", "sum"), ratings=("score", "size"), mean_score=("score", "mean"))
    .reset_index()
)

,blind_id,candidate_id,strategy,rater_id,score
0,WBV-0951233DD6,lapa12b,sequential,rater_01,5.0
1,WBV-0951233DD6,lapa12b,sequential,rater_02,5.0
2,WBV-E69D51AECC,mamay27b,zero_shot,rater_01,3.0
3,WBV-E69D51AECC,mamay27b,zero_shot,rater_02,5.0
4,WBV-5F8CEC8B13,lapa12b,sequential,rater_01,4.0
5,WBV-5F8CEC8B13,lapa12b,sequential,rater_02,5.0
6,WBV-52435814E5,mamay27b,zero_shot,rater_01,3.0
7,WBV-52435814E5,mamay27b,zero_shot,rater_02,5.0
8,WBV-A810A8D6FA,mamay27b,sequential,rater_01,4.0
9,WBV-A810A8D6FA,mamay27b,sequential,rater_02,4.0


,rater_id,items,ratings
0,rater_01,48,48
1,rater_02,48,48


,candidate_id,strategy,score_sum,ratings,mean_score
0,lapa12b,sequential,107.0,24,4.458333
1,lapa12b,zero_shot,107.0,24,4.458333
2,mamay27b,sequential,98.0,24,4.083333
3,mamay27b,zero_shot,88.0,24,3.666667


In [3]:
from warbias_kit.dataset_workflow import recorded_selection

analysis, selection_record = recorded_selection(project, inputs)
columns = [
    "rank",
    "candidate_id",
    "strategy",
    "generated_items",
    "rated_items",
    "ratings",
    "mean_score",
]
display(DataFrame(analysis["condition_summary"])[columns])
analysis["coverage"]

,rank,candidate_id,strategy,generated_items,rated_items,ratings,mean_score
0,1,lapa12b,sequential,13,12,24,4.458333
1,1,lapa12b,zero_shot,12,12,24,4.458333
2,3,mamay27b,sequential,12,12,24,4.083333
3,4,mamay27b,zero_shot,13,12,24,3.666667


{'generated_items': 50,
 'rated_items': 48,
 'ratings': 96,
 'raters': 2,
 'expected_ratings': 100,
 'missing_condition_ids': ['ssuk_idp_return_integration_choice_case_001__idp_current__neutral__unspecified::lapa12b::sequential',
  'ssuk_idp_employment_competence_reliability_case_005__idp_current__neutral__unspecified::mamay27b::zero_shot']}

The arithmetic mean uses both overall quality scores for each reviewed item.
Lapa has the highest observed mean with both prompting methods: 107 / 24 = 4.4583.
The scores do not distinguish sequential from zero-shot prompting.
The recorded author choice is Lapa with sequential prompting for control over
the three sentence roles. This choice is separate from the tied score ranking.

In [4]:
write_json(work / "rating-analysis.json", analysis)
write_json(work / "selected-condition.json", selection_record)
result = {
    "stage": "model_selection",
    "selection_record_path": str(work / "selected-condition.json"),
    "selection": analysis["recorded_selection"],
    "highest_mean_conditions": analysis["highest_mean_conditions"],
    "coverage": analysis["coverage"],
}
result

{'stage': 'model_selection',
 'selection_record_path': 'runs/walkthrough/reviewed-release/02_model_selection-zjfd8n_1/selected-condition.json',
 'selection': {'candidate_id': 'lapa12b',
  'strategy': 'sequential',
  'mean_score': 4.458333333333333,
  'rank': 1,
  'record_type': 'reconstructed_from_contemporaneous_author_instructions_and_frozen_run'},
 'highest_mean_conditions': [{'candidate_id': 'lapa12b',
   'strategy': 'sequential',
   'generated_items': 13,
   'rated_items': 12,
   'ratings': 24,
   'mean_score': 4.458333333333333,
   'missing_condition_ids': ['ssuk_idp_return_integration_choice_case_001__idp_current__neutral__unspecified::lapa12b::sequential'],
   'rank': 1},
  {'candidate_id': 'lapa12b',
   'strategy': 'zero_shot',
   'generated_items': 12,
   'rated_items': 12,
   'ratings': 24,
   'mean_score': 4.458333333333333,
   'missing_condition_ids': [],
   'rank': 1}],
 'coverage': {'generated_items': 50,
  'rated_items': 48,
  'ratings': 96,
  'raters': 2,
  'expected_r

The saved `selected-condition.json` contains the recorded author choice and
hashes of the exact ratings and pilot inputs. Pass its displayed relative path
to `SELECTION_RECORD_PATH` in notebook 3. A tie in the observed means does not
turn the separate author choice into a unique score winner.
